# DML Project: Structural Robustness & Generalization
# Results Visualization Notebook

This notebook visualizes the results from the 4-phase Deep Metric Learning experiment.

## Overview

This project investigates how deep metric representations generalize to entirely unseen classes, addressing severe structural bottlenecks in standard training paradigms.

### Research Questions

1. **The Overfitting Paradox**: Do high-capacity models resist overfitting in retrieval tasks?
2. **The Greediness of Loss**: Does Contrastive Loss induce dimensional collapse vs Triplet Loss?
3. **Memory-Linear Scaling**: Can Shadow Loss achieve equal/better clustering than Triplet?
4. **Synthetic Supervision**: Does L2A-NC yield better generalization than mining?

---

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")

# Configuration
PROJECT_DIR = "/Users/glasslab/dml-project"
RESULTS_DIR = f"{PROJECT_DIR}/results"

print("✓ Imports successful")
print(f"✓ Results directory: {RESULTS_DIR}")

## 1. Phase 1: Zero-Shot Baseline Evaluation

Evaluate CLIP, DINO, and ResNet backbones without training.

In [ ]:
# Load Phase 1 results
phase1_path = f"{RESULTS_DIR}/phase1_baseline.json"

if os.path.exists(phase1_path):
    with open(phase1_path, 'r') as f:
        phase1_results = json.load(f)
    print("✓ Phase 1 results loaded")
    print(f"  Backbones evaluated: {list(phase1_results.keys())}")
else:
    print("⚠ Phase 1 results not found. Run `python scripts/run_pipeline.py --phase 1`")
    phase1_results = {}

# Display metrics
def display_metrics(results, title="Metrics"):
    print(f"\n{title}:")
    for backbone, data in results.items():
        print(f"\n  {backbone}:")
        for split, metrics in data.items():
            print(f"    {split}:")
            for metric, value in metrics.items():
                if isinstance(value, float):
                    print(f"      {metric}: {value:.4f}")

In [ ]:
# Display Phase 1 results
display_metrics(phase1_results, "Phase 1: Zero-Shot Baseline Evaluation")

In [ ]:
# Visualize Phase 1 results
def plot_phase1_comparison(phase1_results):
    if not phase1_results:
        print("⚠ No Phase 1 results to plot")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Phase 1: Zero-Shot Baseline Comparison', fontsize=16, fontweight='bold')
    
    metrics_to_plot = ['nmi', 'ari', 'silhouette', 'opis']
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        backbones = list(phase1_results.keys())
        test_seen_values = []
        test_unseen_values = []
        
        for backbone in backbones:
            if 'test_seen' in phase1_results[backbone]:
                test_seen_values.append(phase1_results[backbone]['test_seen'].get(metric, 0))
            else:
                test_seen_values.append(0)
                
            if 'test_unseen' in phase1_results[backbone]:
                test_unseen_values.append(phase1_results[backbone]['test_unseen'].get(metric, 0))
            else:
                test_unseen_values.append(0)
        
        x = np.arange(len(backbones))
        width = 0.35
        
        ax.bar(x - width/2, test_seen_values, width, label='Test-seen', alpha=0.8)
        ax.bar(x + width/2, test_unseen_values, width, label='Test-unseen', alpha=0.8)
        
        ax.set_ylabel(metric.upper())
        ax.set_title(f'{metric.upper()} Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(backbones, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/phase1_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_phase1_comparison(phase1_results)

## 2. Phase 2: Training Evaluation

Evaluate Contrastive, Triplet, and Shadow Loss training.

In [ ]:
# Load Phase 2 results
phase2_contrastive_path = f"{RESULTS_DIR}/phase2_contrastive_results.json"
phase2_triplet_path = f"{RESULTS_DIR}/phase2_triplet_results.json"
phase2_shadow_path = f"{RESULTS_DIR}/phase2_shadow_results.json"

phase2_results = {}

for path, name in [(phase2_contrastive_path, 'contrastive'), 
                   (phase2_triplet_path, 'triplet'),
                   (phase2_shadow_path, 'shadow')]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            phase2_results[name] = json.load(f)
        print(f"✓ Phase 2 ({name}) results loaded")
    else:
        print(f"⚠ Phase 2 ({name}) results not found")

def display_phase2_results(phase2_results):
    print("\nPhase 2: Training Results")
    print("-" * 50)
    
    for loss_type, results in phase2_results.items():
        print(f"\n{loss_type.upper()}:")
        print(f"  Best Val Score: {results.get('best_val_score', 0):.4f}")
        
        if 'test_seen' in results:
            print(f"  Test-seen Grouped Recall@K: {results['test_seen'].get('grouped_recall_at_k', 0):.4f}")
            print(f"  Test-seen OPIS: {results['test_seen'].get('opis', 0):.4f}")
            
        if 'test_unseen' in results:
            print(f"  Test-unseen Grouped Recall@K: {results['test_unseen'].get('grouped_recall_at_k', 0):.4f}")
            print(f"  Test-unseen OPIS: {results['test_unseen'].get('opis', 0):.4f}")
        
        if 'generalization_gap' in results:
            print(f"  Generalization Gap:")
            gap = results['generalization_gap']
            print(f"    Grouped Recall@K: {gap.get('grouped_recall_at_k', 0):.4f}")
            print(f"    OPIS: {gap.get('opis', 0):.4f}")

In [ ]:
display_phase2_results(phase2_results)

In [ ]:
# Visualize Phase 2 comparison
def plot_phase2_comparison(phase2_results):
    if not phase2_results:
        print("⚠ No Phase 2 results to plot")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Phase 2: Training Loss Comparison', fontsize=16, fontweight='bold')
    
    metrics_to_plot = ['grouped_recall_at_k', 'opis', 'silhouette', 'ami']
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        losses = list(phase2_results.keys())
        test_seen_values = []
        test_unseen_values = []
        
        for loss in losses:
            if 'test_seen' in phase2_results[loss]:
                test_seen_values.append(phase2_results[loss]['test_seen'].get(metric, 0))
            else:
                test_seen_values.append(0)
                
            if 'test_unseen' in phase2_results[loss]:
                test_unseen_values.append(phase2_results[loss]['test_unseen'].get(metric, 0))
            else:
                test_unseen_values.append(0)
        
        x = np.arange(len(losses))
        width = 0.35
        
        ax.bar(x - width/2, test_seen_values, width, label='Test-seen', alpha=0.8)
        ax.bar(x + width/2, test_unseen_values, width, label='Test-unseen', alpha=0.8)
        
        ax.set_ylabel(metric.upper())
        ax.set_title(f'{metric.upper()} Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(losses, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/phase2_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_phase2_comparison(phase2_results)

## 3. Phase 4: Combined Analysis & Statistical Validation

Final evaluation with statistical significance testing.

In [ ]:
# Load Phase 4 results
phase4_path = f"{RESULTS_DIR}/phase4_final_results.json"

if os.path.exists(phase4_path):
    with open(phase4_path, 'r') as f:
        phase4_results = json.load(f)
    print("✓ Phase 4 results loaded")
else:
    print("⚠ Phase 4 results not found. Run full pipeline first.")
    phase4_results = {}

In [ ]:
# Comprehensive comparison plot
def plot_comprehensive_comparison(phase4_results):
    if not phase4_results:
        print("⚠ No Phase 4 results to plot")
        return
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    fig.suptitle('Phase 4: Comprehensive Evaluation & Statistical Validation', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    metrics_to_plot = ['grouped_recall_at_k', 'opis', 'nmi', 'ari', 'ami', 'silhouette']
    
    # Get all methods
    methods = ['clip_vit_base_patch32', 'contrastive', 'triplet', 'shadow']
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        seen_values = []
        unseen_values = []
        method_names = []
        
        for method in methods:
            if method in phase4_results['baseline']:
                seen_values.append(phase4_results['baseline'][method].get('test_seen', {}).get(metric, 0))
                unseen_values.append(phase4_results['baseline'][method].get('test_unseen', {}).get(metric, 0))
                method_names.append(method)
            elif method in phase4_results:
                seen_values.append(phase4_results[method].get('test_seen', {}).get(metric, 0))
                unseen_values.append(phase4_results[method].get('test_unseen', {}).get(metric, 0))
                method_names.append(method)
        
        x = np.arange(len(methods))
        width = 0.35
        
        ax.bar(x - width/2, seen_values, width, label='Test-seen', alpha=0.8)
        ax.bar(x + width/2, unseen_values, width, label='Test-unseen', alpha=0.8)
        
        ax.set_ylabel(metric.upper())
        ax.set_title(f'{metric.upper()} Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(method_names, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/phase4_comprehensive.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_comprehensive_comparison(phase4_results)

In [ ]:
# Display statistical test results
def display_statistical_tests(phase4_results):
    print("\nPhase 4: Statistical Tests")
    print("-" * 50)
    
    if 'statistical_tests' in phase4_results:
        for test_name, test_results in phase4_results['statistical_tests'].items():
            print(f"\n{test_name}:")
            print(f"  T-statistic: {test_results.get('t_statistic', 0):.4f}")
            print(f"  P-value: {test_results.get('p_value', 0):.4f}")
            print(f"  Significant (α=0.05): {test_results.get('significant', False)}")
    else:
        print("⚠ No statistical tests found")

In [ ]:
display_statistical_tests(phase4_results)

## 4. UMAP Visualizations

Visualize embedding spaces from different methods.

In [ ]:
# Display UMAP plots
umap_dir = f"{RESULTS_DIR}/umap"

if os.path.exists(umap_dir):
    print("UMAP Plots:")
    for file in os.listdir(umap_dir):
        print(f"  - {file}")
    
    # Show plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('UMAP Embedding Spaces', fontsize=14, fontweight='bold')
    
    plots = ['umap_test_seen.png', 'umap_test_unseen.png', 'umap_combined.png']
    titles = ['Test-seen (80 classes)', 'Test-unseen (20 classes)', 'Combined (100 classes)']
    
    for idx, (plot, title) in enumerate(zip(plots, titles)):
        plot_path = f"{umap_dir}/{plot}"
        if os.path.exists(plot_path):
            img = plt.imread(plot_path)
            axes[idx].imshow(img)
            axes[idx].set_title(title)
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/umap_overview.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ UMAP plots not found. Run full pipeline.")

## 5. Summary & Conclusions

Key findings from the experiment.

In [ ]:
# Generate summary table
def generate_summary(phase4_results):
    print("\n" + "="*60)
    print("SUMMARY: Key Findings")
    print("="*60)
    
    if not phase4_results:
        print("⚠ Run full pipeline to generate summary")
        return
    
    # Compare methods on test-unseen grouped_recall_at_k
    print("\n1. Generalization to Unseen Classes (Test-unseen Grouped Recall@K):")
    print("-" * 60)
    
    methods = ['clip_vit_base_patch32', 'contrastive', 'triplet', 'shadow']
    recall_values = []
    
    for method in methods:
        if method in phase4_results['baseline']:
            recall = phase4_results['baseline'][method].get('test_unseen', {}).get('grouped_recall_at_k', 0)
        elif method in phase4_results:
            recall = phase4_results[method].get('test_unseen', {}).get('grouped_recall_at_k', 0)
        else:
            recall = 0
        recall_values.append(recall)
        print(f"  {method:30s}: {recall:.4f}")
    
    # Find best method
    best_idx = np.argmax(recall_values)
    print(f"\n  ✓ Best generalization: {methods[best_idx]}")
    
    # Check OPIS (threshold consistency)
    print("\n2. Threshold Consistency (OPIS - lower is better):")
    print("-" * 60)
    
    opis_values = []
    for method in methods:
        if method in phase4_results['baseline']:
            opis = phase4_results['baseline'][method].get('test_unseen', {}).get('opis', 0)
        elif method in phase4_results:
            opis = phase4_results[method].get('test_unseen', {}).get('opis', 0)
        else:
            opis = 0
        opis_values.append(opis)
        print(f"  {method:30s}: {opis:.4f}")
    
    best_opis_idx = np.argmin(opis_values)
    print(f"\n  ✓ Best threshold consistency: {methods[best_opis_idx]}")
    
    # Statistical significance
    print("\n3. Statistical Validation:")
    print("-" * 60)
    
    if 'statistical_tests' in phase4_results:
        for test_name, test_results in phase4_results['statistical_tests'].items():
            significant = test_results.get('significant', False)
            p_value = test_results.get('p_value', 0)
            print(f"  {test_name}:")
            print(f"    P-value: {p_value:.4f}")
            print(f"    Significant (α=0.05): {significant}")
    else:
        print("  ⚠ No statistical tests found")
    
    print("\n" + "="*60)

generate_summary(phase4_results)

In [ ]:
# Save summary to file
def save_summary(phase4_results, output_path):
    summary = []
    summary.append("="*60)
    summary.append("DML Project: Structural Robustness & Generalization")
    summary.append("Summary Report")
    summary.append("="*60)
    summary.append("")
    
    if phase4_results:
        methods = ['clip_vit_base_patch32', 'contrastive', 'triplet', 'shadow']
        
        summary.append("1. Generalization to Unseen Classes:")
        summary.append("-" * 50)
        for method in methods:
            if method in phase4_results['baseline']:
                recall = phase4_results['baseline'][method].get('test_unseen', {}).get('grouped_recall_at_k', 0)
            elif method in phase4_results:
                recall = phase4_results[method].get('test_unseen', {}).get('grouped_recall_at_k', 0)
            else:
                recall = 0
            summary.append(f"  {method:30s}: {recall:.4f}")
        summary.append("")
        
        summary.append("2. Threshold Consistency (OPIS):")
        summary.append("-" * 50)
        for method in methods:
            if method in phase4_results['baseline']:
                opis = phase4_results['baseline'][method].get('test_unseen', {}).get('opis', 0)
            elif method in phase4_results:
                opis = phase4_results[method].get('test_unseen', {}).get('opis', 0)
            else:
                opis = 0
            summary.append(f"  {method:30s}: {opis:.4f}")
        summary.append("")
        
        if 'statistical_tests' in phase4_results:
            summary.append("3. Statistical Validation:")
            summary.append("-" * 50)
            for test_name, test_results in phase4_results['statistical_tests'].items():
                summary.append(f"  {test_name}:")
                summary.append(f"    P-value: {test_results.get('p_value', 0):.4f}")
                summary.append(f"    Significant: {test_results.get('significant', False)}")
            summary.append("")
    
    summary.append("="*60)
    summary.append("End of Report")
    summary.append("="*60)
    
    with open(output_path, 'w') as f:
        f.write("\n".join(summary))
    
    print(f"✓ Summary saved to {output_path}")

save_summary(phase4_results, f"{RESULTS_DIR}/summary_report.txt")